In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import os
import torch
import os
import time
import pickle
import matplotlib.pyplot as plt
from scipy.stats import norm

from e_1_run_cvae import train_chunk, alarm
from e_1_run_cvae_time_check import train_chunk_time_check
#from send_result import send_result
#from e_2_CVAE_norm import CVAE as CVAE_norm
#from e_1_run_cvae_norm import train_chunk_norm

# global var
S0 = 1.0
K = 1.0
r = 0.03
sigma = np.sqrt(0.05)
T = 1.5
BS_eta = (S0, K, r, sigma, T)
# S0, K, r, kappa, theta, xi, rho, Y0, T = Hes_eta
Hes_eta = (S0, K, r, 2, 0.05, 0.5, -0.7, 0.05, T)

B = 0.8 # down-and-out must B < S0 and B < K
model_type = 'hes_clip' # bs, bs_clip, hes, hes_clip
# barr_type = 'van' # van or barr
# opt_type = 'call' # call or put
# chunk_dir = f"/mnt/d/bs_chunks_correction/" if model_type == 'bs' else f"/mnt/d/hes_chunks_correction/"
# eta_path = "/mnt/d/bs_eta_basic.h5" if model_type == 'bs' else "/mnt/d/hes_eta_basic.h5"

if not((B < S0) & (B < K)):
    raise ValueError("down-and-out : B should be smaller than S0 and K")

# if not(opt_type == 'call' or  opt_type == 'put'):
#     raise ValueError("option_type must be 'call' or 'put'")

# if not(barr_type == 'van' or  barr_type == 'barr'):
#     raise ValueError("barr_type must be 'van' or 'barr'")

if not(model_type == 'hes' or  model_type == 'bs' or model_type == 'bs_clip' or model_type == 'hes_clip'):
    raise ValueError("model_type must be 'hes' or 'bs'")

if not(torch.cuda.is_available()):
    raise ValueError("CUDA is not available")
device = torch.device("cuda")

bs_stats = {
    "x_mean": -0.1045446063,
    "x_std": 0.6455563393,
    "m_mean": -0.4579059199,
    "m_std": 0.5553496410
}

if model_type == 'hes':
    test_etas = [0.03, 2.0,  0.05, 0.5, -0.7, 0.05, 1.5]
    eta_keys  = ['r', 'lambda', 'v_bar', 'xi', 'rho', 'Y0', 'T']
else: # model_type = 'bs'
    test_etas = [r, sigma, T]
    eta_keys  = ['r', 'sigma', 'T']

# if model_type == 'hes':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.115733
#         else: # put
#             bench_price = 0.005170
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.124491
#         else: # put
#             bench_price = 0.080488

# elif model_type == 'bs':
#     if barr_type == 'barr':
#         if opt_type == 'call':
#             bench_price = 0.123493
#         else: # put
#             bench_price = 0.009535
#     else: # van
#         if opt_type == 'call':
#             bench_price = 0.129944
#         else: # put
#             bench_price = 0.085942

/home/enjongoopee/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


# training

In [2]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 1024, 1024, 512, 256] # [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024], [1024, 1024, 1024, 1024, 512, 256]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 11
lr2         = 2e-6
l2          = 12
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 190 # 1 chunk train : 2m
validation_chunk_idxs = [22,64,76,116,155,194]
val_every_chunks = 194
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = None # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True

init_path = None
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk190.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [3]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0, 
    K=K, 
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=0->190 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/194 | file_idx  81 | BN off    | beta_eff: 1.0000 | Recon: -2.1872 | KL: 1.1996 | Total: -0.9875
Chunk step     2 | epoch    1 chunk   2/194 | file_idx 124 | BN off    | beta_eff: 1.0000 | Recon: -2.5456 | KL: 1.3614 | Total: -1.1842
Chunk step     3 | epoch    1 chunk   3/194 | file_idx  62 | BN off    | beta_eff: 1.0000 | Recon: -2.6533 | KL: 1.4068 | Total: -1.2464
Chunk step     4 | epoch    1 chunk   4/194 | file_idx  32 | BN off    | beta_eff: 1.0000 | Recon: -2.6912 | KL: 1.4014 | Total: -1.2898
Chunk step     5 | epoch    1 chunk   5/194 | file_idx  86 | BN off    | beta_eff: 1.0000 | Recon: -2.7854 | KL: 1.4509 | Total: -1.3345
Chunk step     6 | epoch    

In [4]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk190.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk190.pt | 완료 chunks=190
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=190->194 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   191 | epoch    1 chunk 191/194 | file_idx 174 | BN off    | beta_eff: 1.0000 | Recon: -4.5931 | KL: 3.0081 | Total: -1.5850
Validation 시작 @ chunk 191
Validation @ chunk   191 | Recon: -4.6019 | KL: 3.0253 | Total: -1.5766 | KL_dim: [0.888817, 2.136455]
Chunk step   192 | epoch    1 chunk 192/194 | file_idx 179 | BN off    | beta_eff: 1.0000 | Recon: -4.6107 | KL: 3.0363 | Total: -1.5744
Validation 시작 @ chunk 192
Validation @ chunk   192 | Recon: -4.6219 | KL: 3.0384 | Total: -1.5834 | KL_dim: [0.898495, 2.139948]
Chunk step   193 | epoch    1 chunk 193/194 | 

In [5]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk194.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk384.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk194.pt | 완료 chunks=194
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=194->384 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   195 | epoch    2 chunk   1/194 | file_idx 182 | BN off    | beta_eff: 1.0000 | Recon: -4.6027 | KL: 3.0240 | Total: -1.5787
Chunk step   196 | epoch    2 chunk   2/194 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -4.6105 | KL: 3.0249 | Total: -1.5856
Chunk step   197 | epoch    2 chunk   3/194 | file_idx  84 | BN off    | beta_eff: 1.0000 | Recon: -4.6027 | KL: 3.0205 | Total: -1.5822
Chunk step   198 | epoch    2 chunk   4/194 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -4.6029 | KL: 3.0148 | Total: -1.5881
Chunk step   199 | epoch  

In [6]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk384.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk384.pt | 완료 chunks=384
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=384->388 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base


Chunk step   385 | epoch    2 chunk 191/194 | file_idx 102 | BN off    | beta_eff: 1.0000 | Recon: -4.7488 | KL: 3.1582 | Total: -1.5906
Validation 시작 @ chunk 385
Validation @ chunk   385 | Recon: -4.7622 | KL: 3.1728 | Total: -1.5893 | KL_dim: [0.973872, 2.198969]
Chunk step   386 | epoch    2 chunk 192/194 | file_idx 103 | BN off    | beta_eff: 1.0000 | Recon: -4.7534 | KL: 3.1684 | Total: -1.5851
Validation 시작 @ chunk 386
Validation @ chunk   386 | Recon: -4.7668 | KL: 3.1738 | Total: -1.5930 | KL_dim: [0.97514, 2.198706]
Chunk step   387 | epoch    2 chunk 193/194 | file_idx  74 | BN off    | beta_eff: 1.0000 | Recon: -4.7553 | KL: 3.1610 | Total: -1.5944
Validation 시작 @ chunk 387
Validation @ chunk   387 | Recon: -4.7359 | KL: 3.1396 | Total: -1.5964 | KL_dim: [0.955877, 2.183709]
Chunk step   388 | epoch    2 chunk 194/194 | file_idx 199 | BN off    | beta_eff: 1.0000 | Recon: -4.7506 | KL: 3.1682 | Total: -1.5825
Validation 시작 @ chunk 388
Validation @ chunk   388 | Recon: -4.758

In [7]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk388.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk578.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk388.pt | 완료 chunks=388
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=388->578 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   389 | epoch    3 chunk   1/194 | file_idx 197 | BN off    | beta_eff: 1.0000 | Recon: -4.7565 | KL: 3.1654 | Total: -1.5910
Chunk step   390 | epoch    3 chunk   2/194 | file_idx  29 | BN off    | beta_eff: 1.0000 | Recon: -4.7460 | KL: 3.1666 | Total: -1.5794
Chunk step   391 | epoch    3 chunk   3/194 | file_idx 150 | BN off    | beta_eff: 1.0000 | Recon: -4.7561 | KL: 3.1651 | Total: -1.5910
Chunk step   392 | epoch    3 chunk   4/194 | file_idx  33 | BN off    | beta_eff: 1.0000 | Recon: -4.7642 | KL: 3.1736 | Total: -1.5906
Chunk step   393 | epoch  

In [8]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk578.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk578.pt | 완료 chunks=578
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=578->582 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   579 | epoch    3 chunk 191/194 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -4.8175 | KL: 3.2251 | Total: -1.5923
Validation 시작 @ chunk 579
Validation @ chunk   579 | Recon: -4.8196 | KL: 3.2238 | Total: -1.5959 | KL_dim: [1.000583, 2.223192]
Chunk step   580 | epoch    3 chunk 192/194 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -4.8075 | KL: 3.2218 | Total: -1.5857
Validation 시작 @ chunk 580
Validation @ chunk   580 | Recon: -4.8217 | KL: 3.2333 | Total: -1.5884 | KL_dim: [1.004848, 2.228432]
Chunk step   581 | epoch    3 chunk 193/194 | 

In [9]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk582.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk772.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk582.pt | 완료 chunks=582
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=582->772 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   583 | epoch    4 chunk   1/194 | file_idx 150 | BN off    | beta_eff: 1.0000 | Recon: -4.8275 | KL: 3.2340 | Total: -1.5935
Chunk step   584 | epoch    4 chunk   2/194 | file_idx 148 | BN off    | beta_eff: 1.0000 | Recon: -4.8139 | KL: 3.2278 | Total: -1.5861
Chunk step   585 | epoch    4 chunk   3/194 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -4.8143 | KL: 3.2218 | Total: -1.5924
Chunk step   586 | epoch    4 chunk   4/194 | file_idx 126 | BN off    | beta_eff: 1.0000 | Recon: -4.8186 | KL: 3.2232 | Total: -1.5954
Chunk step   587 | epoch  

In [10]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk772.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk772.pt | 완료 chunks=772
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=772->776 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   773 | epoch    4 chunk 191/194 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -4.8535 | KL: 3.2665 | Total: -1.5870
Validation 시작 @ chunk 773
Validation @ chunk   773 | Recon: -4.8540 | KL: 3.2579 | Total: -1.5962 | KL_dim: [1.020697, 2.237147]
Chunk step   774 | epoch    4 chunk 192/194 | file_idx 136 | BN off    | beta_eff: 1.0000 | Recon: -4.8508 | KL: 3.2583 | Total: -1.5925
Validation 시작 @ chunk 774
Validation @ chunk   774 | Recon: -4.8588 | KL: 3.2606 | Total: -1.5982 | KL_dim: [1.020737, 2.239841]
Chunk step   775 | epoch    4 chunk 193/194 | 

In [11]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk776.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk966.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk776.pt | 완료 chunks=776
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=776->966 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   777 | epoch    5 chunk   1/194 | file_idx 113 | BN off    | beta_eff: 1.0000 | Recon: -4.8550 | KL: 3.2636 | Total: -1.5914
Chunk step   778 | epoch    5 chunk   2/194 | file_idx 118 | BN off    | beta_eff: 1.0000 | Recon: -4.8526 | KL: 3.2574 | Total: -1.5952
Chunk step   779 | epoch    5 chunk   3/194 | file_idx 163 | BN off    | beta_eff: 1.0000 | Recon: -4.8512 | KL: 3.2575 | Total: -1.5937
Chunk step   780 | epoch    5 chunk   4/194 | file_idx 132 | BN off    | beta_eff: 1.0000 | Recon: -4.8526 | KL: 3.2601 | Total: -1.5925
Chunk step   781 | epoch  

In [12]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk966.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk966.pt | 완료 chunks=966
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=966->970 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   967 | epoch    5 chunk 191/194 | file_idx 122 | BN off    | beta_eff: 1.0000 | Recon: -4.8762 | KL: 3.2810 | Total: -1.5953
Validation 시작 @ chunk 967
Validation @ chunk   967 | Recon: -4.8878 | KL: 3.2924 | Total: -1.5953 | KL_dim: [1.037444, 2.254972]
Chunk step   968 | epoch    5 chunk 192/194 | file_idx  65 | BN off    | beta_eff: 1.0000 | Recon: -4.8808 | KL: 3.2849 | Total: -1.5959
Validation 시작 @ chunk 968
Validation @ chunk   968 | Recon: -4.8908 | KL: 3.2982 | Total: -1.5927 | KL_dim: [1.041618, 2.256533]
Chunk step   969 | epoch    5 chunk 193/194 | 

In [13]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1160.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk970.pt | 완료 chunks=970
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=970->1160 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step   971 | epoch    6 chunk   1/194 | file_idx 109 | BN off    | beta_eff: 1.0000 | Recon: -4.8782 | KL: 3.2896 | Total: -1.5886
Chunk step   972 | epoch    6 chunk   2/194 | file_idx 186 | BN off    | beta_eff: 1.0000 | Recon: -4.8701 | KL: 3.2784 | Total: -1.5917
Chunk step   973 | epoch    6 chunk   3/194 | file_idx  59 | BN off    | beta_eff: 1.0000 | Recon: -4.8878 | KL: 3.3022 | Total: -1.5855
Chunk step   974 | epoch    6 chunk   4/194 | file_idx  75 | BN off    | beta_eff: 1.0000 | Recon: -4.8804 | KL: 3.2902 | Total: -1.5902
Chunk step   975 | epoch 

In [14]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1160.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1160.pt | 완료 chunks=1160
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=1160->1164 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1161 | epoch    6 chunk 191/194 | file_idx  33 | BN off    | beta_eff: 1.0000 | Recon: -4.9088 | KL: 3.3145 | Total: -1.5943
Validation 시작 @ chunk 1161
Validation @ chunk  1161 | Recon: -4.9243 | KL: 3.3302 | Total: -1.5941 | KL_dim: [1.056481, 2.273677]
Chunk step  1162 | epoch    6 chunk 192/194 | file_idx 104 | BN off    | beta_eff: 1.0000 | Recon: -4.9100 | KL: 3.3112 | Total: -1.5988
Validation 시작 @ chunk 1162
Validation @ chunk  1162 | Recon: -4.9196 | KL: 3.3235 | Total: -1.5961 | KL_dim: [1.050167, 2.273361]
Chunk step  1163 | epoch    6 chunk 193/

In [15]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1164.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1354.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1164.pt | 완료 chunks=1164
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=1164->1354 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1165 | epoch    7 chunk   1/194 | file_idx  55 | BN off    | beta_eff: 1.0000 | Recon: -4.8992 | KL: 3.3108 | Total: -1.5884
Chunk step  1166 | epoch    7 chunk   2/194 | file_idx 127 | BN off    | beta_eff: 1.0000 | Recon: -4.9074 | KL: 3.3141 | Total: -1.5933
Chunk step  1167 | epoch    7 chunk   3/194 | file_idx 129 | BN off    | beta_eff: 1.0000 | Recon: -4.9072 | KL: 3.3219 | Total: -1.5853
Chunk step  1168 | epoch    7 chunk   4/194 | file_idx 163 | BN off    | beta_eff: 1.0000 | Recon: -4.9109 | KL: 3.3156 | Total: -1.5953
Chunk step  1169 | epo

In [16]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1354.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1354.pt | 완료 chunks=1354
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=1354->1358 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1355 | epoch    7 chunk 191/194 | file_idx 193 | BN off    | beta_eff: 1.0000 | Recon: -4.9412 | KL: 3.3464 | Total: -1.5948
Validation 시작 @ chunk 1355
Validation @ chunk  1355 | Recon: -4.9384 | KL: 3.3411 | Total: -1.5974 | KL_dim: [1.0559, 2.285176]
Chunk step  1356 | epoch    7 chunk 192/194 | file_idx 153 | BN off    | beta_eff: 1.0000 | Recon: -4.9348 | KL: 3.3410 | Total: -1.5938
Validation 시작 @ chunk 1356
Validation @ chunk  1356 | Recon: -4.9534 | KL: 3.3553 | Total: -1.5981 | KL_dim: [1.061287, 2.294035]
Chunk step  1357 | epoch    7 chunk 193/19

In [17]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1358.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1548.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1358.pt | 완료 chunks=1358
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=1358->1548 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1359 | epoch    8 chunk   1/194 | file_idx  89 | BN off    | beta_eff: 1.0000 | Recon: -4.9310 | KL: 3.3414 | Total: -1.5895
Chunk step  1360 | epoch    8 chunk   2/194 | file_idx 171 | BN off    | beta_eff: 1.0000 | Recon: -4.9394 | KL: 3.3429 | Total: -1.5965
Chunk step  1361 | epoch    8 chunk   3/194 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -4.9324 | KL: 3.3440 | Total: -1.5884
Chunk step  1362 | epoch    8 chunk   4/194 | file_idx  80 | BN off    | beta_eff: 1.0000 | Recon: -4.9356 | KL: 3.3405 | Total: -1.5951
Chunk step  1363 | epo

In [18]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1548.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1548.pt | 완료 chunks=1548
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=1548->1552 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1549 | epoch    8 chunk 191/194 | file_idx  72 | BN off    | beta_eff: 1.0000 | Recon: -4.9823 | KL: 3.3944 | Total: -1.5879
Validation 시작 @ chunk 1549
Validation @ chunk  1549 | Recon: -4.9825 | KL: 3.3841 | Total: -1.5983 | KL_dim: [1.06694, 2.317185]
Chunk step  1550 | epoch    8 chunk 192/194 | file_idx 152 | BN off    | beta_eff: 1.0000 | Recon: -4.9811 | KL: 3.3865 | Total: -1.5945
Validation 시작 @ chunk 1550
Validation @ chunk  1550 | Recon: -4.9875 | KL: 3.3923 | Total: -1.5951 | KL_dim: [1.075642, 2.31671]
Chunk step  1551 | epoch    8 chunk 193/19

In [19]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1552.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1742.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1552.pt | 완료 chunks=1552
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=1552->1742 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1553 | epoch    9 chunk   1/194 | file_idx 133 | BN off    | beta_eff: 1.0000 | Recon: -4.9720 | KL: 3.3822 | Total: -1.5898
Chunk step  1554 | epoch    9 chunk   2/194 | file_idx  62 | BN off    | beta_eff: 1.0000 | Recon: -4.9803 | KL: 3.3840 | Total: -1.5964
Chunk step  1555 | epoch    9 chunk   3/194 | file_idx  95 | BN off    | beta_eff: 1.0000 | Recon: -4.9824 | KL: 3.3822 | Total: -1.6001
Chunk step  1556 | epoch    9 chunk   4/194 | file_idx 103 | BN off    | beta_eff: 1.0000 | Recon: -4.9794 | KL: 3.3888 | Total: -1.5906
Chunk step  1557 | epo

In [20]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1742.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1746.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1742.pt | 완료 chunks=1742
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=1742->1746 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step  1743 | epoch    9 chunk 191/194 | file_idx 196 | BN off    | beta_eff: 1.0000 | Recon: -5.0439 | KL: 3.4397 | Total: -1.6042
Validation 시작 @ chunk 1743
Validation @ chunk  1743 | Recon: -5.0350 | KL: 3.4461 | Total: -1.5889 | KL_dim: [1.084112, 2.361968]
Chunk step  1744 | epoch    9 chunk 192/194 | file_idx  70 | BN off    | beta_eff: 1.0000 | Recon: -5.0350 | KL: 3.4462 | Total: -1.5888
Validation 시작 @ chunk 1744
Validation @ chunk  1744 | Recon: -5.0422 | KL: 3.4507 | Total: -1.5915 | KL_dim: [1.090197, 2.360551]
Chunk step  1745 | epoch    9 chunk 193/

In [3]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1746.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1936.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1746.pt | 완료 chunks=1746
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=1746->1936 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1747 | epoch   10 chunk   1/194 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.0467 | KL: 3.4431 | Total: -1.6036
Chunk step  1748 | epoch   10 chunk   2/194 | file_idx   8 | BN off    | beta_eff: 1.0000 | Recon: -5.0486 | KL: 3.4506 | Total: -1.5981
Chunk step  1749 | epoch   10 chunk   3/194 | file_idx 149 | BN off    | beta_eff: 1.0000 | Recon: -5.0430 | KL: 3.4456 | Total: -1.5974
Chunk step  1750 | epoch   10 chunk   4/194 | file_idx  11 | BN off    | beta_eff: 1.0000 | Recon: -5.0395 | KL: 3.4344 | Total: -1.

In [4]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1936.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1940.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1936.pt | 완료 chunks=1936
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=1936->1940 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1937 | epoch   10 chunk 191/194 | file_idx 154 | BN off    | beta_eff: 1.0000 | Recon: -5.0936 | KL: 3.4987 | Total: -1.5949
Validation 시작 @ chunk 1937
Validation @ chunk  1937 | Recon: -5.0898 | KL: 3.4894 | Total: -1.6003 | KL_dim: [1.089644, 2.399801]
Chunk step  1938 | epoch   10 chunk 192/194 | file_idx  66 | BN off    | beta_eff: 1.0000 | Recon: -5.1038 | KL: 3.5066 | Total: -1.5973
Validation 시작 @ chunk 1938
Validation @ chunk  1938 | Recon: -5.1040 | KL: 3.5026 | Total: -1.6014 | KL_dim: [1.096254, 2.406298]
Chunk step  1

In [5]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1940.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2130.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk1940.pt | 완료 chunks=1940
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=1940->2130 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  1941 | epoch   11 chunk   1/194 | file_idx 111 | BN off    | beta_eff: 1.0000 | Recon: -5.0987 | KL: 3.5034 | Total: -1.5952
Chunk step  1942 | epoch   11 chunk   2/194 | file_idx 133 | BN off    | beta_eff: 1.0000 | Recon: -5.1021 | KL: 3.5104 | Total: -1.5917
Chunk step  1943 | epoch   11 chunk   3/194 | file_idx 105 | BN off    | beta_eff: 1.0000 | Recon: -5.1025 | KL: 3.5009 | Total: -1.6016
Chunk step  1944 | epoch   11 chunk   4/194 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -5.1002 | KL: 3.5022 | Total: -1.

In [6]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2130.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2134.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2130.pt | 완료 chunks=2130
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=2130->2134 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2131 | epoch   11 chunk 191/194 | file_idx 142 | BN off    | beta_eff: 1.0000 | Recon: -5.1670 | KL: 3.5719 | Total: -1.5951
Validation 시작 @ chunk 2131
Validation @ chunk  2131 | Recon: -5.1695 | KL: 3.5730 | Total: -1.5965 | KL_dim: [1.115885, 2.457119]
Chunk step  2132 | epoch   11 chunk 192/194 | file_idx 106 | BN off    | beta_eff: 1.0000 | Recon: -5.1713 | KL: 3.5777 | Total: -1.5935
Validation 시작 @ chunk 2132
Validation @ chunk  2132 | Recon: -5.1725 | KL: 3.5753 | Total: -1.5972 | KL_dim: [1.113789, 2.461541]
Chunk step  2

In [7]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2134.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2324.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2134.pt | 완료 chunks=2134
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=2134->2324 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2135 | epoch   12 chunk   1/194 | file_idx 184 | BN off    | beta_eff: 1.0000 | Recon: -5.1686 | KL: 3.5751 | Total: -1.5934
Chunk step  2136 | epoch   12 chunk   2/194 | file_idx  83 | BN off    | beta_eff: 1.0000 | Recon: -5.1721 | KL: 3.5712 | Total: -1.6009
Chunk step  2137 | epoch   12 chunk   3/194 | file_idx  45 | BN off    | beta_eff: 1.0000 | Recon: -5.1801 | KL: 3.5713 | Total: -1.6088
Chunk step  2138 | epoch   12 chunk   4/194 | file_idx 178 | BN off    | beta_eff: 1.0000 | Recon: -5.1756 | KL: 3.5807 | Total: -1.

In [8]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2324.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2328.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2324.pt | 완료 chunks=2324
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=2324->2328 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2325 | epoch   12 chunk 191/194 | file_idx 163 | BN off    | beta_eff: 1.0000 | Recon: -5.2669 | KL: 3.6668 | Total: -1.6000
Validation 시작 @ chunk 2325
Validation @ chunk  2325 | Recon: -5.2340 | KL: 3.6418 | Total: -1.5922 | KL_dim: [1.134067, 2.507693]
Chunk step  2326 | epoch   12 chunk 192/194 | file_idx 173 | BN off    | beta_eff: 1.0000 | Recon: -5.2656 | KL: 3.6631 | Total: -1.6025
Validation 시작 @ chunk 2326
Validation @ chunk  2326 | Recon: -5.2667 | KL: 3.6692 | Total: -1.5975 | KL_dim: [1.146921, 2.522244]
Chunk step  2

In [9]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2328.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2518.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2328.pt | 완료 chunks=2328
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=2328->2518 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2329 | epoch   13 chunk   1/194 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.2539 | KL: 3.6618 | Total: -1.5921
Chunk step  2330 | epoch   13 chunk   2/194 | file_idx 170 | BN off    | beta_eff: 1.0000 | Recon: -5.2548 | KL: 3.6585 | Total: -1.5963
Chunk step  2331 | epoch   13 chunk   3/194 | file_idx  28 | BN off    | beta_eff: 1.0000 | Recon: -5.2650 | KL: 3.6667 | Total: -1.5982
Chunk step  2332 | epoch   13 chunk   4/194 | file_idx  58 | BN off    | beta_eff: 1.0000 | Recon: -5.2592 | KL: 3.6658 | Total: -1.

In [10]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2518.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2522.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2518.pt | 완료 chunks=2518
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=2518->2522 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2519 | epoch   13 chunk 191/194 | file_idx 196 | BN off    | beta_eff: 1.0000 | Recon: -5.3965 | KL: 3.7898 | Total: -1.6068
Validation 시작 @ chunk 2519
Validation @ chunk  2519 | Recon: -5.3856 | KL: 3.7849 | Total: -1.6008 | KL_dim: [1.177615, 2.607243]
Chunk step  2520 | epoch   13 chunk 192/194 | file_idx 190 | BN off    | beta_eff: 1.0000 | Recon: -5.3982 | KL: 3.7959 | Total: -1.6023
Validation 시작 @ chunk 2520
Validation @ chunk  2520 | Recon: -5.4121 | KL: 3.8101 | Total: -1.6020 | KL_dim: [1.185875, 2.624254]
Chunk step  2

In [11]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2522.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2712.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2522.pt | 완료 chunks=2522
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=2522->2712 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2523 | epoch   14 chunk   1/194 | file_idx  31 | BN off    | beta_eff: 1.0000 | Recon: -5.3990 | KL: 3.8012 | Total: -1.5978
Chunk step  2524 | epoch   14 chunk   2/194 | file_idx  52 | BN off    | beta_eff: 1.0000 | Recon: -5.4144 | KL: 3.8081 | Total: -1.6064
Chunk step  2525 | epoch   14 chunk   3/194 | file_idx 119 | BN off    | beta_eff: 1.0000 | Recon: -5.4071 | KL: 3.8107 | Total: -1.5964
Chunk step  2526 | epoch   14 chunk   4/194 | file_idx  61 | BN off    | beta_eff: 1.0000 | Recon: -5.4071 | KL: 3.8138 | Total: -1.

In [12]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2712.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2716.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2712.pt | 완료 chunks=2712
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=2712->2716 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2713 | epoch   14 chunk 191/194 | file_idx  50 | BN off    | beta_eff: 1.0000 | Recon: -5.5184 | KL: 3.9144 | Total: -1.6040
Validation 시작 @ chunk 2713
Validation @ chunk  2713 | Recon: -5.5358 | KL: 3.9365 | Total: -1.5993 | KL_dim: [1.225297, 2.711181]
Chunk step  2714 | epoch   14 chunk 192/194 | file_idx  67 | BN off    | beta_eff: 1.0000 | Recon: -5.5192 | KL: 3.9255 | Total: -1.5938
Validation 시작 @ chunk 2714
Validation @ chunk  2714 | Recon: -5.5048 | KL: 3.9015 | Total: -1.6034 | KL_dim: [1.213167, 2.688291]
Chunk step  2

In [13]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2716.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2906.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2716.pt | 완료 chunks=2716
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=2716->2906 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2717 | epoch   15 chunk   1/194 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.5144 | KL: 3.9139 | Total: -1.6006
Chunk step  2718 | epoch   15 chunk   2/194 | file_idx  12 | BN off    | beta_eff: 1.0000 | Recon: -5.5246 | KL: 3.9182 | Total: -1.6064
Chunk step  2719 | epoch   15 chunk   3/194 | file_idx  77 | BN off    | beta_eff: 1.0000 | Recon: -5.5272 | KL: 3.9285 | Total: -1.5988
Chunk step  2720 | epoch   15 chunk   4/194 | file_idx  73 | BN off    | beta_eff: 1.0000 | Recon: -5.5283 | KL: 3.9302 | Total: -1.

In [14]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2906.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2910.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2906.pt | 완료 chunks=2906
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=2906->2910 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2907 | epoch   15 chunk 191/194 | file_idx 182 | BN off    | beta_eff: 1.0000 | Recon: -5.6078 | KL: 4.0130 | Total: -1.5948
Validation 시작 @ chunk 2907
Validation @ chunk  2907 | Recon: -5.5999 | KL: 4.0035 | Total: -1.5964 | KL_dim: [1.23907, 2.764447]
Chunk step  2908 | epoch   15 chunk 192/194 | file_idx 125 | BN off    | beta_eff: 1.0000 | Recon: -5.6062 | KL: 4.0096 | Total: -1.5965
Validation 시작 @ chunk 2908
Validation @ chunk  2908 | Recon: -5.6229 | KL: 4.0164 | Total: -1.6065 | KL_dim: [1.245269, 2.771149]
Chunk step  29

In [15]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk2910.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3100.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk2910.pt | 완료 chunks=2910
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=2910->3100 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  2911 | epoch   16 chunk   1/194 | file_idx  71 | BN off    | beta_eff: 1.0000 | Recon: -5.6106 | KL: 4.0025 | Total: -1.6081
Chunk step  2912 | epoch   16 chunk   2/194 | file_idx  69 | BN off    | beta_eff: 1.0000 | Recon: -5.6027 | KL: 4.0116 | Total: -1.5911
Chunk step  2913 | epoch   16 chunk   3/194 | file_idx 162 | BN off    | beta_eff: 1.0000 | Recon: -5.6051 | KL: 4.0052 | Total: -1.6000
Chunk step  2914 | epoch   16 chunk   4/194 | file_idx  46 | BN off    | beta_eff: 1.0000 | Recon: -5.6086 | KL: 4.0121 | Total: -1.

In [16]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3100.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3104.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3100.pt | 완료 chunks=3100
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=3100->3104 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3101 | epoch   16 chunk 191/194 | file_idx 149 | BN off    | beta_eff: 1.0000 | Recon: -5.6511 | KL: 4.0483 | Total: -1.6028
Validation 시작 @ chunk 3101
Validation @ chunk  3101 | Recon: -5.6624 | KL: 4.0609 | Total: -1.6015 | KL_dim: [1.263638, 2.797255]
Chunk step  3102 | epoch   16 chunk 192/194 | file_idx 113 | BN off    | beta_eff: 1.0000 | Recon: -5.6532 | KL: 4.0531 | Total: -1.6001
Validation 시작 @ chunk 3102
Validation @ chunk  3102 | Recon: -5.6517 | KL: 4.0465 | Total: -1.6051 | KL_dim: [1.257211, 2.789321]
Chunk step  3

In [17]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3104.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3294.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3104.pt | 완료 chunks=3104
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=3104->3294 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3105 | epoch   17 chunk   1/194 | file_idx 153 | BN off    | beta_eff: 1.0000 | Recon: -5.6440 | KL: 4.0436 | Total: -1.6004
Chunk step  3106 | epoch   17 chunk   2/194 | file_idx  93 | BN off    | beta_eff: 1.0000 | Recon: -5.6490 | KL: 4.0445 | Total: -1.6045
Chunk step  3107 | epoch   17 chunk   3/194 | file_idx  82 | BN off    | beta_eff: 1.0000 | Recon: -5.6557 | KL: 4.0497 | Total: -1.6061
Chunk step  3108 | epoch   17 chunk   4/194 | file_idx 154 | BN off    | beta_eff: 1.0000 | Recon: -5.6486 | KL: 4.0489 | Total: -1.

In [18]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3294.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3298.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3294.pt | 완료 chunks=3294
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=3294->3298 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3295 | epoch   17 chunk 191/194 | file_idx  85 | BN off    | beta_eff: 1.0000 | Recon: -5.6623 | KL: 4.0626 | Total: -1.5998
Validation 시작 @ chunk 3295
Validation @ chunk  3295 | Recon: -5.6890 | KL: 4.0856 | Total: -1.6034 | KL_dim: [1.265615, 2.819993]
Chunk step  3296 | epoch   17 chunk 192/194 | file_idx  78 | BN off    | beta_eff: 1.0000 | Recon: -5.6646 | KL: 4.0679 | Total: -1.5967
Validation 시작 @ chunk 3296
Validation @ chunk  3296 | Recon: -5.6787 | KL: 4.0723 | Total: -1.6064 | KL_dim: [1.259992, 2.812355]
Chunk step  3

In [3]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3298.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3488.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3298.pt | 완료 chunks=3298
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=3298->3488 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3299 | epoch   18 chunk   1/194 | file_idx 180 | BN off    | beta_eff: 1.0000 | Recon: -5.6563 | KL: 4.0543 | Total: -1.6020
Chunk step  3300 | epoch   18 chunk   2/194 | file_idx 141 | BN off    | beta_eff: 1.0000 | Recon: -5.6679 | KL: 4.0626 | Total: -1.6053
Chunk step  3301 | epoch   18 chunk   3/194 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -5.6582 | KL: 4.0699 | Total: -1.5883
Chunk step  3302 | epoch   18 chunk   4/194 | file_idx  60 | BN off    | beta_eff: 1.0000 | Recon: -5.6691 | KL: 4.0632 | Total: -1.

In [4]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3488.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3492.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3488.pt | 완료 chunks=3488
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=3488->3492 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3489 | epoch   18 chunk 191/194 | file_idx 166 | BN off    | beta_eff: 1.0000 | Recon: -5.6713 | KL: 4.0749 | Total: -1.5964
Validation 시작 @ chunk 3489
Validation @ chunk  3489 | Recon: -5.6503 | KL: 4.0451 | Total: -1.6053 | KL_dim: [1.245892, 2.799177]
Chunk step  3490 | epoch   18 chunk 192/194 | file_idx   9 | BN off    | beta_eff: 1.0000 | Recon: -5.6758 | KL: 4.0792 | Total: -1.5966
Validation 시작 @ chunk 3490
Validation @ chunk  3490 | Recon: -5.6667 | KL: 4.0609 | Total: -1.6058 | KL_dim: [1.252299, 2.808614]
Chunk step  3

In [5]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3492.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3582.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3492.pt | 완료 chunks=3492
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=3492->3682 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3493 | epoch   19 chunk   1/194 | file_idx 167 | BN off    | beta_eff: 1.0000 | Recon: -5.6772 | KL: 4.0842 | Total: -1.5930
Chunk step  3494 | epoch   19 chunk   2/194 | file_idx 135 | BN off    | beta_eff: 1.0000 | Recon: -5.6809 | KL: 4.0745 | Total: -1.6063
Chunk step  3495 | epoch   19 chunk   3/194 | file_idx  54 | BN off    | beta_eff: 1.0000 | Recon: -5.6792 | KL: 4.0898 | Total: -1.5893
Chunk step  3496 | epoch   19 chunk   4/194 | file_idx  72 | BN off    | beta_eff: 1.0000 | Recon: -5.6759 | KL: 4.0809 | Total: -1.

In [6]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3582.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3586.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3582.pt | 완료 chunks=3682
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=3682->3686 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3683 | epoch   19 chunk 191/194 | file_idx 109 | BN off    | beta_eff: 1.0000 | Recon: -5.6798 | KL: 4.0821 | Total: -1.5977
Validation 시작 @ chunk 3683
Validation @ chunk  3683 | Recon: -5.6587 | KL: 4.0846 | Total: -1.5741 | KL_dim: [1.264023, 2.820621]
Chunk step  3684 | epoch   19 chunk 192/194 | file_idx 105 | BN off    | beta_eff: 1.0000 | Recon: -5.6863 | KL: 4.0794 | Total: -1.6070
Validation 시작 @ chunk 3684
Validation @ chunk  3684 | Recon: -5.7025 | KL: 4.1015 | Total: -1.6010 | KL_dim: [1.264981, 2.836492]
Chunk step  3

In [7]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3586.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3776.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3586.pt | 완료 chunks=3686
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=3686->3876 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3687 | epoch   20 chunk   1/194 | file_idx 189 | BN off    | beta_eff: 1.0000 | Recon: -5.6812 | KL: 4.0857 | Total: -1.5955
Chunk step  3688 | epoch   20 chunk   2/194 | file_idx 162 | BN off    | beta_eff: 1.0000 | Recon: -5.6827 | KL: 4.0812 | Total: -1.6015
Chunk step  3689 | epoch   20 chunk   3/194 | file_idx 119 | BN off    | beta_eff: 1.0000 | Recon: -5.6887 | KL: 4.0889 | Total: -1.5999
Chunk step  3690 | epoch   20 chunk   4/194 | file_idx 165 | BN off    | beta_eff: 1.0000 | Recon: -5.6891 | KL: 4.0838 | Total: -1.

In [8]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3776.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3780.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3776.pt | 완료 chunks=3876
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=3876->3880 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3877 | epoch   20 chunk 191/194 | file_idx 150 | BN off    | beta_eff: 1.0000 | Recon: -5.6938 | KL: 4.0891 | Total: -1.6047
Validation 시작 @ chunk 3877
Validation @ chunk  3877 | Recon: -5.6814 | KL: 4.0770 | Total: -1.6044 | KL_dim: [1.256197, 2.820797]
Chunk step  3878 | epoch   20 chunk 192/194 | file_idx 117 | BN off    | beta_eff: 1.0000 | Recon: -5.6911 | KL: 4.0951 | Total: -1.5960
Validation 시작 @ chunk 3878
Validation @ chunk  3878 | Recon: -5.6789 | KL: 4.0763 | Total: -1.6026 | KL_dim: [1.25524, 2.821072]
Chunk step  38

In [9]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3780.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3970.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3780.pt | 완료 chunks=3880
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=3880->4070 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  3881 | epoch   21 chunk   1/194 | file_idx 120 | BN off    | beta_eff: 1.0000 | Recon: -5.6937 | KL: 4.0865 | Total: -1.6072
Chunk step  3882 | epoch   21 chunk   2/194 | file_idx  72 | BN off    | beta_eff: 1.0000 | Recon: -5.6864 | KL: 4.0911 | Total: -1.5953
Chunk step  3883 | epoch   21 chunk   3/194 | file_idx 192 | BN off    | beta_eff: 1.0000 | Recon: -5.6913 | KL: 4.0998 | Total: -1.5916
Chunk step  3884 | epoch   21 chunk   4/194 | file_idx 199 | BN off    | beta_eff: 1.0000 | Recon: -5.6786 | KL: 4.0835 | Total: -1.

In [10]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3970.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk3974.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk3970.pt | 완료 chunks=4070
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=4070->4074 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4071 | epoch   21 chunk 191/194 | file_idx 126 | BN off    | beta_eff: 1.0000 | Recon: -5.7002 | KL: 4.0938 | Total: -1.6064
Validation 시작 @ chunk 4071
Validation @ chunk  4071 | Recon: -5.6911 | KL: 4.0854 | Total: -1.6057 | KL_dim: [1.257054, 2.828311]
Chunk step  4072 | epoch   21 chunk 192/194 | file_idx 168 | BN off    | beta_eff: 1.0000 | Recon: -5.7065 | KL: 4.0965 | Total: -1.6101
Validation 시작 @ chunk 4072
Validation @ chunk  4072 | Recon: -5.7018 | KL: 4.0983 | Total: -1.6035 | KL_dim: [1.25939, 2.838944]
Chunk step  40

In [5]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4074.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4264.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk4074.pt | 완료 chunks=4074
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=4074->4264 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4075 | epoch   22 chunk   1/194 | file_idx 102 | BN off    | beta_eff: 1.0000 | Recon: -5.6952 | KL: 4.0914 | Total: -1.6038
Chunk step  4076 | epoch   22 chunk   2/194 | file_idx 115 | BN off    | beta_eff: 1.0000 | Recon: -5.6955 | KL: 4.0923 | Total: -1.6033
Chunk step  4077 | epoch   22 chunk   3/194 | file_idx 160 | BN off    | beta_eff: 1.0000 | Recon: -5.6970 | KL: 4.0923 | Total: -1.6047
Chunk step  4078 | epoch   22 chunk   4/194 | file_idx 134 | BN off    | beta_eff: 1.0000 | Recon: -5.7025 | KL: 4.0943 | Total: -1.

In [6]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4264.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4268.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk4264.pt | 완료 chunks=4264
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=4264->4268 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4265 | epoch   22 chunk 191/194 | file_idx 175 | BN off    | beta_eff: 1.0000 | Recon: -5.7028 | KL: 4.0988 | Total: -1.6040
Validation 시작 @ chunk 4265
Validation @ chunk  4265 | Recon: -5.7107 | KL: 4.1080 | Total: -1.6027 | KL_dim: [1.270721, 2.837302]
Chunk step  4266 | epoch   22 chunk 192/194 | file_idx 165 | BN off    | beta_eff: 1.0000 | Recon: -5.7057 | KL: 4.0994 | Total: -1.6063
Validation 시작 @ chunk 4266
Validation @ chunk  4266 | Recon: -5.7149 | KL: 4.1094 | Total: -1.6056 | KL_dim: [1.262321, 2.847042]
Chunk step  4

In [7]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4268.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4458.pt"


time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk4268.pt | 완료 chunks=4268
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=4268->4458 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4269 | epoch   23 chunk   1/194 | file_idx  37 | BN off    | beta_eff: 1.0000 | Recon: -5.6966 | KL: 4.0934 | Total: -1.6032
Chunk step  4270 | epoch   23 chunk   2/194 | file_idx 111 | BN off    | beta_eff: 1.0000 | Recon: -5.7014 | KL: 4.1001 | Total: -1.6012
Chunk step  4271 | epoch   23 chunk   3/194 | file_idx 156 | BN off    | beta_eff: 1.0000 | Recon: -5.7034 | KL: 4.0990 | Total: -1.6043
Chunk step  4272 | epoch   23 chunk   4/194 | file_idx  10 | BN off    | beta_eff: 1.0000 | Recon: -5.6996 | KL: 4.0951 | Total: -1.

In [8]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4458.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4462.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk4458.pt | 완료 chunks=4458
learning rate : 2e-05
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=4458->4462 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4459 | epoch   23 chunk 191/194 | file_idx  33 | BN off    | beta_eff: 1.0000 | Recon: -5.7129 | KL: 4.1094 | Total: -1.6035
Validation 시작 @ chunk 4459
Validation @ chunk  4459 | Recon: -5.7086 | KL: 4.1101 | Total: -1.5985 | KL_dim: [1.262447, 2.847665]
Chunk step  4460 | epoch   23 chunk 192/194 | file_idx 149 | BN off    | beta_eff: 1.0000 | Recon: -5.7127 | KL: 4.1085 | Total: -1.6042
Validation 시작 @ chunk 4460
Validation @ chunk  4460 | Recon: -5.6909 | KL: 4.0873 | Total: -1.6036 | KL_dim: [1.252523, 2.834733]
Chunk step  4

In [ ]:
# CVAE training settings
dim_z       = 2 # 2, 4, 8
hidden_dims = [1024, 1024, 1024, 1024, 512, 256] # [1024, 1024, 512, 256], [1024, 1024, 1024, 1024, 1024], [1024, 1024, 1024, 1024, 512, 256]
batch_size  = 16384 # 1024, 2048, 128-4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 2e-5 # 수렴속도 1e-4 < 1e-5
l1          = 26
lr2         = 2e-6
l2          = 27
lr3         = 1e-6
l3          = 13
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 190 # 1 chunk train : 2m
validation_chunk_idxs = [22,64,76,116,155,194]
val_every_chunks = 194
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"(exponential), "normal_weight", "add_put_loss"
weight_mode = None # barr_weight : "barrier_put", "barrier_near" / normal_weight, add_put_loss: "barrier_put", "all_put"
weight_alpha = 1.0 # 1.0, 2.0, 3.0 
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True

init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk4462.pt" # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4652.pt"
if 'base' in save_path:
    raise ValueError("base는 save_path에 포함되면 안됨.")

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

In [4]:
time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_2e-05_1_[22, 64, 76, 116, 155, 194]_chunk4462.pt | 완료 chunks=4462
learning rate : 2e-06
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=4462->4652 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4463 | epoch   24 chunk   1/194 | file_idx  62 | BN off    | beta_eff: 1.0000 | Recon: -5.7908 | KL: 4.1801 | Total: -1.6107
Chunk step  4464 | epoch   24 chunk   2/194 | file_idx 187 | BN off    | beta_eff: 1.0000 | Recon: -5.8413 | KL: 4.2321 | Total: -1.6092
Chunk step  4465 | epoch   24 chunk   3/194 | file_idx 185 | BN off    | beta_eff: 1.0000 | Recon: -5.8616 | KL: 4.2489 | Total: -1.6127
Chunk step  4466 | epoch   24 chunk   4/194 | file_idx 166 | BN off    | beta_eff: 1.0000 | Recon: -5.8723 | KL: 4.2674 | Total: -1.

In [5]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4652.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4656.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_11-2e-05_12-2e-06_1_[22, 64, 76, 116, 155, 194]_chunk4652.pt | 완료 chunks=4652
learning rate : 2e-06
학습 시작 | 이번 실행 chunks=4 | 진행 chunks=4652->4656 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=1 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4653 | epoch   24 chunk 191/194 | file_idx 188 | BN off    | beta_eff: 1.0000 | Recon: -6.0201 | KL: 4.4082 | Total: -1.6119
Validation 시작 @ chunk 4653
Validation @ chunk  4653 | Recon: -6.0218 | KL: 4.4095 | Total: -1.6123 | KL_dim: [1.349901, 3.059628]
Chunk step  4654 | epoch   24 chunk 192/194 | file_idx  90 | BN off    | beta_eff: 1.0000 | Recon: -6.0190 | KL: 4.4079 | Total: -1.6111
Validation 시작 @ chunk 4654
Validation @ chunk  4654 | Recon: -6.0194 | KL: 4.4071 | Total: -1.6123 | KL_dim: [1.349628, 3.057513]
C

In [ ]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4656.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4846.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

체크포인트 재개: result/cvae/hes_clip/cvae_hes_clip_2_1024_1024_1024_16384_None_11-2e-05_12-2e-06_1_[22, 64, 76, 116, 155, 194]_chunk4656.pt | 완료 chunks=4656
learning rate : 2e-06
학습 시작 | 이번 실행 chunks=190 | 진행 chunks=4656->4846 | files/epoch=194 | excluded=[22, 64, 76, 116, 155, 194] | validation=[22, 64, 76, 116, 155, 194] | val_every_chunks=194 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base | tmp_save_every_chunks=10
Chunk step  4657 | epoch   25 chunk   1/194 | file_idx 175 | BN off    | beta_eff: 1.0000 | Recon: -6.0188 | KL: 4.4061 | Total: -1.6127
Chunk step  4658 | epoch   25 chunk   2/194 | file_idx  25 | BN off    | beta_eff: 1.0000 | Recon: -6.0205 | KL: 4.4080 | Total: -1.6125
Chunk step  4659 | epoch   25 chunk   3/194 | file_idx 159 | BN off    | beta_eff: 1.0000 | Recon: -6.0186 | KL: 4.4137 | Total: -1.6049
Chunk step  4660 | epoch   25 chunk   4/194 | file_idx  56 | BN off    | beta_eff: 1.0000 | Recon: -6.0194 | KL: 4.4090 

In [ ]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4846.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4850.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk4850.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5040.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5040.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5044.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 190
val_every_chunks = 194
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5044.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5234.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

In [ ]:
# CVAE training settings
num_chunks  = 4
val_every_chunks = 1
init_path = None
resume_path = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5234.pt" # or None 이어서 학습하고 싶을 
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{l1}-{lr}_{l2}-{lr2}_{beta}_{validation_chunk_idxs}_chunk5238.pt"

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    init_path=init_path,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

# time check

In [3]:
dim_z       = 2 # 8, 12
hidden_dims = [1024, 1024, 1024, 1024] # [128, 128, 64], [256, 256, 128], [512, 256, 128], [1024, 512, 256], [2048, 1024, 512], [1024, 512, 256, 128]
batch_size  = 16384 # 1024, 2048, 4096, 8192, 16384, 32768, 65536
bn_chunks   = None # None or num
use_bn      = True if bn_chunks is not None else False
lr          = 1e-5 # 수렴속도 1e-4 < 1e-5
l1          = 5
lr2         = 1e-5
l2          = 6
lr3         = 1e-6
l3          = 4
beta        = 1
warmup_chunks = None # None or num
num_chunks  = 1 # 1 chunk train : 2m
validation_chunk_idxs = [15,24,78]
val_every_chunks = 97
memory_on_gpu = True
cvae_type = "base" # "base" or "barr_weight"
weight_mode = "barrier_put" # "barrier_put" or "barrier_near"
weight_alpha = 3.0
weight_h = 0.01 # 0.01, 0.03, 0.04
weight_normalize = True
resume_path = None # or None 이어서 학습하고 싶을 때
save_path   = f"result/cvae/{model_type}/cvae_{model_type}_{dim_z}_{hidden_dims[0]}_{hidden_dims[1]}_{hidden_dims[2]}\
_{batch_size}_{bn_chunks}_{lr}_{beta}_{validation_chunk_idxs}_chunk1.pt"

real_lr = lr
if str(lr3) in save_path:
    real_lr = lr3
elif str(lr2) in save_path:
    real_lr = lr2

time1 = time.time()
cvae, loss_history, eta_min, eta_max = train_chunk_time_check(
    model_type=model_type,
    dim_z=dim_z,
    hidden_dims=hidden_dims,
    batch_size=batch_size,
    lr=real_lr,
    beta=beta,
    warmup_chunks=warmup_chunks,
    use_bn=use_bn,
    bn_chunks=bn_chunks,
    num_chunks=num_chunks,
    save_path=save_path,
    resume_path=resume_path,
    validation_chunk_idxs=validation_chunk_idxs,
    val_every_chunks=val_every_chunks,
    memory_on_gpu=memory_on_gpu,
    cvae_type=cvae_type,
    weight_mode=weight_mode,
    weight_alpha=weight_alpha,
    weight_h=weight_h,
    weight_normalize=weight_normalize,
    S0=S0,
    K=K,
    B=B,
)
time2 = time.time()
print(f"Training time: {time2 - time1:.6f}s")

eta min/max 계산 중
계산 완료

learning rate : 1e-05
GPU: NVIDIA GeForce RTX 4080 SUPER | cuda capability=(8, 9)
학습 시작 | 이번 실행 chunks=1 | 진행 chunks=0->1 | files/epoch=97 | excluded=[15, 24, 78] | validation=[15, 24, 78] | val_every_chunks=97 | bn_chunks=None | warmup_chunks=None | memory_on_gpu=True | cvae_type=base
Chunk step     1 | epoch    1 chunk   1/97 | file_idx  88 | BN off    | beta_eff: 1.0000 | Recon: -1.5530 | KL: 1.5218 | Total: -0.0311
Time | total=214.36s | chunk_load=14.70s | load_wait=14.69s | gpu_move=0.15s | loader_init=0.00s | iter_init=0.00s | batch_fetch=0.00s (0.0000/batch) | h2d=0.00s (0.0000/batch) | fwd=72.16s (0.0176/batch) | bwd=132.07s (0.0322/batch) | clip=2.96s (0.0007/batch) | step=6.19s (0.0015/batch) | loss_item=0.87s | cleanup=0.00s | loop_overhead=0.00s (0.0000/batch) | batches=4096
Validation @ chunk     1 | Recon: -2.1291 | KL: 1.8620 | Total: -0.2671 | KL_dim: [0.325633, 1.536353]
Validation time: 267.00s

=== Time summary for this run ===
chunk_load    